In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import utulek

utulek.platform.import_globals_notebook(globals())

base_module = utulek


I0000 00:00:1794242111.916069 2012700 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794242111.924763 2012700 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794242114.465418 2012700 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794242114.468184 2012700 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


IS_NOTEBOOK_KERNEL_CODE = True
NOTEBOOK_NAME = 2026_11_09-transformer.ipynb
ASSET_PATH = /home/gilgamesh/main.syncthing/utulek/experiment/2026_11_09-transformer.ipynb.asset/
Loaded `/home/gilgamesh/main.syncthing/utulek/experiment/.env`.
torch: []
tf: 
jax: [CpuDevice(id=0)]


E0000 00:00:1794242118.205851 2012700 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
BATCH_SIZE = 32
SEQ_LEN = 16
VOCAB_SIZE = 16

In [5]:
starts = torch.randint(VOCAB_SIZE, (BATCH_SIZE, 1))
starts

tensor([[ 7],
        [ 1],
        [ 7],
        [12],
        [15],
        [ 7],
        [10],
        [ 3],
        [12],
        [ 7],
        [ 0],
        [13],
        [14],
        [ 3],
        [15],
        [ 3],
        [ 7],
        [10],
        [14],
        [ 3],
        [15],
        [ 7],
        [ 4],
        [ 9],
        [ 6],
        [ 5],
        [ 6],
        [ 2],
        [12],
        [13],
        [ 9],
        [ 7]])

In [7]:
offsets = torch.arange(SEQ_LEN + 1)
offsets

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16])

In [8]:
tokens = (starts + offsets) % VOCAB_SIZE
tokens

tensor([[ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1],
        [ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12],
        [15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15],
        [ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
        [ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3],
        [12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12],
        [ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0],
        [13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11,

In [9]:
X = tokens[:, :-1]
X

tensor([[ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0],
        [ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6],
        [12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
        [15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
        [ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6],
        [10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2],
        [12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
        [ 7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15],
        [13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12],
        [14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [ 3,  4,  5,  6,  7,  8,  9, 1

In [11]:
Y = tokens[:, 1:]
Y

tensor([[ 8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1],
        [ 8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12],
        [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15],
        [ 8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10],
        [ 4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3],
        [13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12],
        [ 8,  9, 10, 11, 12, 13, 14, 15,  0,  1,  2,  3,  4,  5,  6,  7],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,  0],
        [14, 15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
        [15,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14],
        [ 4,  5,  6,  7,  8,  9, 10, 1

In [12]:
X.shape, Y.shape

(torch.Size([32, 16]), torch.Size([32, 16]))

In [13]:
D_MODEL = 32

In [18]:
token_embedding = torch.nn.Embedding(VOCAB_SIZE, D_MODEL)
token_embedding

Embedding(16, 32)

In [19]:
position_embedding = torch.nn.Embedding(SEQ_LEN, D_MODEL)
position_embedding

Embedding(16, 32)

In [21]:
positions = torch.arange(X.shape[1])
positions

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [ ]:
torch.tensor([[10], [20]]) + torch.tensor([1, 2, 3])

tensor([[11, 12, 13],
        [21, 22, 23]])

In [ ]:
torch.tensor([[10, 20, 30], [40, 50, 60]]) + torch.tensor(
	[1, 2])

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

In [31]:
position_embedding.weight.unsqueeze(1).shape

torch.Size([16, 1, 32])

In [33]:
token_vectors = token_embedding(X)
token_vectors

tensor([[[-0.2859, -0.3896, -1.1569,  ...,  0.8338,  1.1815,  1.2133],
         [-0.0483, -0.0576, -0.9223,  ..., -0.7901, -0.1155,  0.2483],
         [-0.1551,  0.6768, -1.8737,  ...,  0.6001,  1.3768,  1.2918],
         ...,
         [-0.4214,  0.0646,  0.8340,  ..., -1.3609, -1.4735, -0.9134],
         [ 0.6694,  0.4901, -0.6877,  ..., -0.6276, -0.9507, -0.8895],
         [ 0.8904, -1.2464,  0.2382,  ..., -0.4873, -0.3412, -1.0797]],

        [[-1.0976,  1.1889,  0.2951,  ...,  0.0112,  1.6432,  0.7409],
         [-0.2209,  0.3830, -0.0459,  ...,  0.4629,  0.2254, -0.8472],
         [ 0.4378,  0.3024,  0.8852,  ..., -0.4549, -0.2135,  0.6566],
         ...,
         [-1.2703,  0.1091, -2.1122,  ..., -1.1480,  0.4109,  1.5216],
         [ 1.0479, -0.5557, -0.0826,  ...,  1.6494,  1.1707, -1.1971],
         [ 0.5994, -0.7908,  0.5581,  ...,  0.8310,  0.9412,  1.9459]],

        [[-0.2859, -0.3896, -1.1569,  ...,  0.8338,  1.1815,  1.2133],
         [-0.0483, -0.0576, -0.9223,  ..., -0

In [36]:
position_vectors = position_embedding(positions)
position_vectors

tensor([[ 2.7825e+00, -1.1837e+00,  5.3158e-01, -1.0214e+00, -1.6524e+00,
         -7.4517e-01, -1.6381e+00,  7.1574e-01, -2.0296e-01, -2.8524e+00,
         -1.1234e+00, -7.8187e-01,  3.5595e-01,  7.0666e-01, -1.4212e+00,
          8.7666e-01, -1.0517e+00, -1.9245e-01,  7.2272e-01,  1.9571e+00,
          1.0204e+00,  1.3529e+00, -2.5682e-01, -3.6372e-01, -1.5589e+00,
         -1.9322e+00, -1.4955e+00, -7.5561e-01, -6.0382e-02, -7.9564e-01,
         -1.0007e+00, -1.4000e-01],
        [ 1.6998e+00, -4.6885e-01,  6.0719e-01, -8.3510e-01,  1.0003e-01,
          5.0221e-01, -5.3764e-01,  9.9944e-01, -8.7364e-01,  3.9170e-01,
          5.0925e-01, -2.6661e+00, -1.5049e+00, -1.4173e+00,  1.8886e+00,
         -8.4285e-01,  1.9536e+00,  2.5401e-01,  7.6528e-01,  8.7489e-01,
         -9.2245e-01, -5.4831e-01, -1.0113e+00,  1.5898e+00,  1.6808e+00,
          5.3346e-02, -6.8531e-01,  1.3129e+00, -1.9561e-01, -2.2507e+00,
         -1.4539e+00, -3.1772e-01],
        [ 1.0506e+00,  1.3633e+00, -4.87

In [39]:
h = token_vectors + position_vectors
h

tensor([[[ 2.4966, -1.5733, -0.6253,  ...,  0.0381,  0.1808,  1.0733],
         [ 1.6515, -0.5264, -0.3151,  ..., -3.0408, -1.5693, -0.0694],
         [ 0.8955,  2.0401, -2.3615,  ...,  3.3977,  0.8498,  0.6592],
         ...,
         [-0.2121,  1.1619,  0.9249,  ..., -0.7588, -2.6126, -1.2182],
         [-0.2569,  1.3903,  0.3527,  ..., -0.2181, -0.9916, -0.8811],
         [-0.0682, -2.8295, -0.6620,  ...,  0.4731, -1.7338, -0.9353]],

        [[ 1.6849,  0.0052,  0.8267,  ..., -0.7844,  0.6425,  0.6009],
         [ 1.4789, -0.0858,  0.5613,  ..., -1.7878, -1.2284, -1.1649],
         [ 1.4884,  1.6657,  0.3973,  ...,  2.3427, -0.7406,  0.0240],
         ...,
         [-1.0609,  1.2064, -2.0213,  ..., -0.5459, -0.7282,  1.2168],
         [ 0.1216,  0.3444,  0.9578,  ...,  2.0588,  1.1298, -1.1887],
         [-0.3592, -2.3739, -0.3421,  ...,  1.7913, -0.4513,  2.0902]],

        [[ 2.4966, -1.5733, -0.6253,  ...,  0.0381,  0.1808,  1.0733],
         [ 1.6515, -0.5264, -0.3151,  ..., -3

In [40]:
N_HEADS = 4
HEAD_DIM = D_MODEL // N_HEADS
assert HEAD_DIM * N_HEADS == D_MODEL

In [45]:
q_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)
q_proj.weight, q_proj

(Parameter containing:
 tensor([[-0.0002,  0.1538,  0.1553,  ...,  0.0517, -0.0497, -0.1391],
         [-0.0702,  0.1748, -0.0119,  ...,  0.1758, -0.0108, -0.0633],
         [ 0.1326, -0.1724, -0.1180,  ..., -0.1413,  0.0793, -0.1666],
         ...,
         [ 0.1740,  0.1231, -0.1765,  ...,  0.0433, -0.0544, -0.0856],
         [ 0.0936, -0.0263, -0.1467,  ..., -0.1766, -0.0520,  0.1237],
         [ 0.0180,  0.0138, -0.0252,  ...,  0.0841,  0.1372, -0.0894]],
        requires_grad=True),
 Linear(in_features=32, out_features=32, bias=False))

In [46]:
k_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)
v_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)

In [48]:
q = q_proj(h)
k = k_proj(h)
v = v_proj(h)
q.shape

torch.Size([32, 16, 32])

In [49]:
B, T, _ = h.shape

q = q.reshape(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
q.shape

torch.Size([32, 4, 16, 8])

In [50]:
k = k.reshape(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
v = v.reshape(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)

In [52]:
scores = q @ k.transpose(-2, -1)
scores, scores.shape

(tensor([[[[ 7.3729e-01, -2.5273e-01, -5.4217e-01,  ..., -1.2034e+00,
            -8.1679e-01,  1.3443e-02],
           [ 1.8010e+00, -1.6402e+00,  2.3276e+00,  ..., -2.0996e+00,
            -1.6836e-01,  2.2629e+00],
           [ 1.1477e+00,  4.1112e-01,  3.2394e-01,  ..., -2.3218e-01,
            -1.8909e-01,  1.2654e-01],
           ...,
           [ 1.8180e-01,  2.8012e+00, -2.9921e+00,  ...,  1.2677e+00,
             4.9397e-01, -5.9793e-01],
           [-5.9141e-01,  2.8940e+00, -3.2661e+00,  ...,  4.7708e-01,
             5.0956e-01,  1.2568e+00],
           [-2.9423e+00, -2.0156e+00,  6.1707e-01,  ..., -2.0877e+00,
            -8.0574e-01,  2.2790e+00]],
 
          [[-2.3817e+00,  2.1764e+00,  5.3239e+00,  ...,  2.9538e+00,
             3.3584e+00,  2.8403e+00],
           [-2.6393e+00, -1.9340e+00,  4.4524e+00,  ...,  5.1916e-01,
             1.9973e+00,  9.0912e-01],
           [-2.2784e-01,  3.9243e-01, -2.1008e+00,  ..., -9.6046e-01,
            -4.0161e+00, -4.2379e+00],


In [ ]:
scores = scores / (HEAD_DIM**0.5)

In [ ]:
T = q.shape[-2]
future_mask = torch.ones(T, T,
	dtype=torch.bool).triu(diagonal=1)
future_mask

tensor([[False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False,  True,  True,  True,
          True,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False, False,  True,  True,
    

In [56]:
scores = scores.masked_fill(future_mask, float("-inf"))
scores

tensor([[[[ 2.6067e-01,        -inf,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 6.3677e-01, -5.7990e-01,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 4.0577e-01,  1.4535e-01,  1.1453e-01,  ...,        -inf,
                  -inf,        -inf],
          ...,
          [ 6.4277e-02,  9.9039e-01, -1.0579e+00,  ...,  4.4819e-01,
                  -inf,        -inf],
          [-2.0910e-01,  1.0232e+00, -1.1547e+00,  ...,  1.6867e-01,
            1.8016e-01,        -inf],
          [-1.0402e+00, -7.1263e-01,  2.1817e-01,  ..., -7.3811e-01,
           -2.8487e-01,  8.0574e-01]],

         [[-8.4206e-01,        -inf,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [-9.3313e-01, -6.8377e-01,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [-8.0553e-02,  1.3874e-01, -7.4273e-01,  ...,        -inf,
                  -inf,        -inf],
          ...,
     

In [57]:
weights = torch.softmax(scores, dim=-1)
weights

tensor([[[[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.7715, 0.2285, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.3971, 0.3061, 0.2968,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0727, 0.1834, 0.0237,  ..., 0.1067, 0.0000, 0.0000],
          [0.0541, 0.1856, 0.0210,  ..., 0.0790, 0.0799, 0.0000],
          [0.0177, 0.0245, 0.0622,  ..., 0.0239, 0.0376, 0.1119]],

         [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.4380, 0.5620, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.3622, 0.4510, 0.1868,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0812, 0.0314, 0.0420,  ..., 0.0513, 0.0000, 0.0000],
          [0.0416, 0.0692, 0.0725,  ..., 0.0409, 0.0383, 0.0000],
          [0.1950, 0.0331, 0.0089,  ..., 0.0191, 0.0393, 0.0808]],

         [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.9518, 0.0482, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.1120, 0.8514, 0.0366,  ..., 0

In [60]:
head_outputs = weights @ v
head_outputs, head_outputs.shape

(tensor([[[[ 1.3619e+00, -5.7035e-01, -4.6165e-01,  ...,  7.3594e-02,
            -1.3324e+00,  1.7741e-01],
           [ 1.1532e+00, -6.8792e-01, -8.4772e-01,  ...,  5.1594e-02,
            -1.0013e+00,  4.3298e-01],
           [ 7.6082e-01, -8.0716e-01, -7.2838e-01,  ..., -1.7583e-01,
            -3.7341e-01,  7.2936e-01],
           ...,
           [ 4.9906e-01, -4.2741e-01, -5.1413e-01,  ..., -4.5899e-02,
            -1.9932e-01,  3.2204e-01],
           [ 4.7364e-01, -4.4391e-01, -5.5500e-01,  ..., -3.6775e-02,
            -1.9798e-01,  3.9917e-01],
           [ 4.8606e-01, -7.2111e-01, -4.5494e-01,  ..., -3.1162e-01,
            -4.7713e-01,  4.3229e-02]],
 
          [[-7.2193e-01,  3.4704e-01, -1.1350e+00,  ...,  9.2084e-01,
             7.2211e-01,  7.5794e-01],
           [ 1.7862e-01, -9.7484e-01, -1.3544e+00,  ...,  1.0600e+00,
             3.8745e-01, -1.0934e+00],
           [ 1.4895e-01, -6.6488e-01, -8.3823e-01,  ...,  1.1140e+00,
             2.5438e-01, -7.2483e-01],


In [62]:
combined = head_outputs.transpose(1, 2).reshape(B, T, -1)
combined.shape

torch.Size([32, 16, 32])

In [65]:
out_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)
attention_output = out_proj(combined)
attention_output, attention_output.shape

(tensor([[[ 0.2207,  0.2249, -0.6862,  ...,  0.0965,  0.7554, -0.5771],
          [ 0.6077,  0.1024,  0.0139,  ...,  0.3906,  0.5046, -0.8399],
          [ 0.4607, -0.1579,  0.5668,  ...,  0.7889, -0.4386, -0.7713],
          ...,
          [ 0.1954,  0.1631, -0.0888,  ...,  0.0337, -0.0849, -0.3427],
          [ 0.2010,  0.0076, -0.1193,  ...,  0.0535, -0.0027, -0.2150],
          [ 0.1442,  0.0116, -0.2936,  ...,  0.0311,  0.1763, -0.1214]],
 
         [[-0.1484,  0.1668, -0.9391,  ..., -0.1417,  1.0698,  0.0614],
          [-0.1224,  0.2707, -0.0341,  ...,  0.1894,  0.2008, -0.1834],
          [-0.0921,  0.1521,  0.2450,  ...,  0.2853, -0.1962, -0.1642],
          ...,
          [ 0.2275,  0.1584, -0.2114,  ...,  0.0999, -0.0463, -0.3577],
          [ 0.1349,  0.0161, -0.1343,  ...,  0.0783,  0.0816, -0.2292],
          [ 0.0388,  0.0217, -0.2059,  ..., -0.0454,  0.0392, -0.1446]],
 
         [[ 0.2207,  0.2249, -0.6862,  ...,  0.0965,  0.7554, -0.5771],
          [ 0.6077,  0.1024,

In [66]:
norm1 = nn.LayerNorm(D_MODEL)
norm2 = nn.LayerNorm(D_MODEL)

In [67]:
mlp = nn.Sequential(
	nn.Linear(D_MODEL, 4 * D_MODEL),
	nn.GELU(),
	nn.Linear(4 * D_MODEL, D_MODEL),
)

In [68]:
h_attn = norm1(h + attention_output)
h_out = norm2(h_attn + mlp(h_attn))

In [69]:
lm_head = nn.Linear(D_MODEL, VOCAB_SIZE)

logits = lm_head(h_out)
logits, logits.shape

(tensor([[[ 1.0896, -0.8459, -0.1390,  ...,  0.2396,  0.0749, -0.6506],
          [-0.0336, -0.7921, -0.4651,  ..., -0.1295,  0.3487,  0.6361],
          [ 0.5116, -1.0469,  1.5271,  ..., -0.7080,  0.0736, -0.2478],
          ...,
          [-0.8676,  1.0670, -0.7102,  ...,  0.3094,  0.5137,  0.6448],
          [ 0.5851,  0.0326,  0.0135,  ..., -0.3669,  0.5788, -0.8148],
          [-0.5244, -0.5987, -0.0587,  ..., -0.2145, -0.5138, -1.0746]],
 
         [[ 1.4388, -0.5037, -0.4283,  ...,  0.1259,  0.0839, -0.8952],
          [-0.8041, -0.1838,  0.1820,  ...,  0.9440,  1.1642,  0.0071],
          [-0.6527, -0.7018,  0.9867,  ..., -1.0748, -0.5698,  1.3046],
          ...,
          [ 0.2922,  0.4863,  0.5562,  ..., -0.0065,  0.4079, -0.4017],
          [ 1.0133, -0.4731,  0.3121,  ...,  0.2133,  1.0555, -0.8499],
          [-0.0374, -0.7481,  0.4130,  ..., -0.8220, -1.1537, -0.8397]],
 
         [[ 1.0896, -0.8459, -0.1390,  ...,  0.2396,  0.0749, -0.6506],
          [-0.0336, -0.7921,

In [71]:
loss = torch.nn.functional.cross_entropy(
	logits.reshape(-1, VOCAB_SIZE),
	Y.reshape(-1),
)
loss

tensor(2.7996, grad_fn=<NllLossBackward0>)

In [72]:
loss.backward()

In [73]:
print(token_embedding.weight.grad.norm())
print(q_proj.weight.grad.norm())

tensor(0.0976)
tensor(0.0740)


In [ ]:
class TinyTransformer(nn.Module):

	def __init__(self):
		super().__init__()

		self.token_embedding = nn.Embedding(VOCAB_SIZE,
			D_MODEL)
		self.position_embedding = nn.Embedding(
			SEQ_LEN, D_MODEL)

		self.q_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)
		self.k_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)
		self.v_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)
		self.out_proj = nn.Linear(D_MODEL, D_MODEL, bias=False)

		self.norm1 = nn.LayerNorm(D_MODEL)
		self.norm2 = nn.LayerNorm(D_MODEL)
		self.mlp = nn.Sequential(
			nn.Linear(D_MODEL, 4 * D_MODEL),
			nn.GELU(),
			nn.Linear(4 * D_MODEL, D_MODEL),
		)
		self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE)

	def forward(self, x):
		# Your existing computation goes here.
		# Use self.token_embedding, self.q_proj, etc.
		# Derive B and T from x.shape.
		# Return logits shaped [B, T, VOCAB_SIZE].
		B, T = x.shape
		token_one_hot = torch.nn.functional.one_hot(x,
			num_classes=VOCAB_SIZE).to(
			dtype=self.token_embedding.weight.dtype)
		token_vectors = token_one_hot @ self.token_embedding.weight
		position_vectors = self.position_embedding.weight[:T]
		h = token_vectors + position_vectors

		q = h @ self.q_proj.weight.T
		k = h @ self.k_proj.weight.T
		v = h @ self.v_proj.weight.T
		q = q.reshape(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
		k = k.reshape(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
		v = v.reshape(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)

		scores = q @ k.transpose(-2, -1) / HEAD_DIM**0.5
		mask = torch.full((T, T),
			float("-inf"),
			device=scores.device,
			dtype=scores.dtype).triu(1)
		scores = scores + mask
		weights = torch.softmax(scores, dim=-1)
		head_outputs = weights @ v

		combined = head_outputs.transpose(1,
			2).reshape(B, T, D_MODEL)
		attention_output = combined @ self.out_proj.weight.T

		h_attn = self.norm1(h + attention_output)
		h_out = self.norm2(h_attn + self.mlp(h_attn))

		return self.lm_head(h_out)


In [94]:
mask = torch.full((T, T), float("-inf")).triu(1)
mask

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

In [95]:
model = TinyTransformer()

In [96]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()

TinyTransformer(
  (token_embedding): Embedding(16, 32)
  (position_embedding): Embedding(16, 32)
  (q_proj): Linear(in_features=32, out_features=32, bias=False)
  (k_proj): Linear(in_features=32, out_features=32, bias=False)
  (v_proj): Linear(in_features=32, out_features=32, bias=False)
  (out_proj): Linear(in_features=32, out_features=32, bias=False)
  (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  (mlp): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=32, bias=True)
  )
  (lm_head): Linear(in_features=32, out_features=16, bias=True)
)

In [108]:
for step in range(201):
	optimizer.zero_grad()

	logits = model(X)
	# print(logits.shape)
	# print(logits[0][0].tolist())
	loss = torch.nn.functional.cross_entropy(
		logits.reshape(-1, VOCAB_SIZE),
		Y.reshape(-1),
	)

	loss.backward()
	optimizer.step()

	if step % 20 == 0:
		# print(logits.argmax(dim=-1).shape)
		accuracy = (logits.argmax(dim=-1) == Y).float().mean()
		print(step, loss.item(), accuracy.item())

0 0.0011124745942652225 1.0
20 0.001082979142665863 1.0
40 0.0010546084959059954 1.0
60 0.001027330756187439 1.0
80 0.0010010570986196399 1.0
100 0.0009757414809428155 1.0
120 0.0009513635886833072 1.0
140 0.0009278330835513771 1.0
160 0.0009051606175489724 1.0
180 0.0008832791936583817 1.0
200 0.0008621365996077657 1.0


In [110]:
model.eval()

changed_X = X.clone()
changed_X[:, 8:] = (changed_X[:, 8:] + 3) % VOCAB_SIZE

with torch.no_grad():
	original_logits = model(X)
	changed_logits = model(changed_X)

	torch.testing.assert_close(
		original_logits[:, :8],
		changed_logits[:, :8],
	)

In [ ]:
def softmax(z):
	z1 = z - np.max(z, axis=-1, keepdims=True)
	# print(z1)
	z2 = np.exp(z1)
	# print(z2)
	return z2 / np.sum(z2, axis=-1, keepdims=True)

In [135]:
z = np.array([
	[2., 1., 0.],
	[1002., 1001., 1000.],
])

In [136]:
p = softmax(z)
print(p)

assert np.isfinite(p).all()
assert np.allclose(p.sum(axis=-1), 1)
assert np.allclose(p[0], p[1])

[[ 0. -1. -2.]
 [ 0. -1. -2.]]
[[1.         0.36787944 0.13533528]
 [1.         0.36787944 0.13533528]]
[[0.66524096 0.24472847 0.09003057]
 [0.66524096 0.24472847 0.09003057]]
